---
title: Citation impact
description: "Citation impact of the publications of the Department of Electrical and Computer Engineering, Aristotle University of Thessaloniki, in Scopus: totals, h-index, concentration, the most-cited papers and normalised impact by paper characteristics."
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "dece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from dece_stats import config, data
from dece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the Scopus export, partial
AS_OF = config.EXPORT_DATE      # the export date (datetime.date)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dece_stats import metrics, periods, small_groups as sg

pubs = data.load_publications()
meta = data.load_metadata()
cites = pubs["cited_by"]
AS_OF_TEXT = f"{AS_OF.day} {AS_OF:%B %Y}"   # the export date as text: day, month name and year
LAST_FULL = periods.LAST_FULL              # the latest full year (PARTIAL - 1)
P_RECENT = periods.period_of(LAST_FULL)    # the latest five full years
P_PREV = periods.period_of(LAST_FULL - 5)  # the five years before them
RECENT_FROM = periods.bounds(P_RECENT)[0]
TEN_FROM = periods.bounds(P_PREV)[0]       # the latest ten full years are TEN_FROM..LAST_FULL
TEN_YEARS = f"{TEN_FROM}–{LAST_FULL}"
MATURE_TO = LAST_FULL - 2                  # papers with at least two full calendar years of citations

# Age-controlled statistics (within publication years, normalised scores) cover NCS_FROM..MATURE_TO. NCS_FROM is
# the first year from which every year has at least NCS_MIN articles (reviews included, see below) and NCS_MIN
# conference papers, so that each year and type has an average of its own.
CITABLE = ["Article", "Review", "Conference paper"]
NCS_TYPE = {"Review": "Article"}           # the normalisation type of a document type
NCS_MIN = 10
citable = pubs[pubs["doc_type"].isin(CITABLE)]
type_year = (pd.crosstab(citable["year"], citable["doc_type"].replace(NCS_TYPE))
             .reindex(index=range(int(pubs["year"].min()), MATURE_TO + 1), columns=["Article", "Conference paper"],
                      fill_value=0))
thin = type_year.min(axis=1) < NCS_MIN    # a year with fewer than NCS_MIN papers of a type
NCS_FROM = int(thin[thin].index.max()) + 1 if thin.any() else int(type_year.index.min())
NCS_TO = MATURE_TO
COHORT_FROM = NCS_FROM                     # the same years for the statistics within publication years


def fmt_count(value: float) -> str:
    """Citation count or median: no decimals unless it has a fraction (5.5)."""
    return dp.fmt_int(value) if float(value).is_integer() else dp.fmt_num(value, 1)


def join_and(items: list[str]) -> str:
    """'a', 'a and b', 'a, b and c' ('' for no items)."""
    return " and ".join(items) if len(items) <= 2 else ", ".join(items[:-1]) + " and " + items[-1]


# --- Headline indicators: all records, Scopus "Cited by" counts at the export date
n_pubs = len(pubs)
n_partial = int((pubs["year"] == PARTIAL).sum())
total_cites = int(cites.sum())
mean_cites = cites.mean()
median_cites = cites.median()
mean_to_median = mean_cites / median_cites
n_uncited = int((cites == 0).sum())
share_uncited = n_uncited / n_pubs
share_uncited_mature = (pubs.loc[pubs["year"] <= MATURE_TO, "cited_by"] == 0).mean()
h_all = metrics.h_index(cites)
h_next = h_all + 1
n_h, n_h_next = int((cites >= h_all).sum()), int((cites >= h_next).sum())   # papers cited >= h, >= h + 1 times
h_ten = metrics.h_index(pubs.loc[pubs["year"].between(TEN_FROM, LAST_FULL), "cited_by"])
h_recent = metrics.h_index(pubs.loc[pubs["year"].between(*periods.bounds(P_RECENT)), "cited_by"])
n_100 = int((cites >= 100).sum())
share_100 = n_100 / n_pubs
share_cites_100 = cites[cites >= 100].sum() / total_cites

# --- Distribution over citation classes
BIN_EDGES = [0, 1, 5, 10, 25, 50, 100, 250, np.inf]
BIN_LABELS = ["0", "1–4", "5–9", "10–24", "25–49", "50–99", "100–249", "≥250"]
cite_class = pd.cut(cites, bins=BIN_EDGES, labels=BIN_LABELS, right=False)
dist = (pubs.groupby(cite_class, observed=False)["cited_by"]
        .agg(papers="size", citations="sum")
        .rename_axis("cite_class").reset_index())
dist["share_papers"] = dist["papers"] / n_pubs
dist["share_citations"] = dist["citations"] / total_cites
under_10 = dist["cite_class"].isin(["0", "1–4", "5–9"])
share_papers_lt10 = dist.loc[under_10, "share_papers"].sum()
share_cites_lt10 = dist.loc[under_10, "share_citations"].sum()

# --- Concentration: papers ranked from most to least cited
top1_n, top10_n, half_n = (int(np.ceil(f * n_pubs)) for f in (0.01, 0.10, 0.50))
top1_share = metrics.top_share(cites, frac=0.01)
top10_share = metrics.top_share(cites, frac=0.10)
half_share = metrics.top_share(cites, frac=0.50)
gini = metrics.gini(cites)
ranked = np.sort(cites.to_numpy())[::-1]
cum_share = np.concatenate([[0.0], np.cumsum(ranked) / ranked.sum()])   # share held by the top k papers
n_half_cites = int(np.searchsorted(cum_share, 0.5))                      # fewest papers holding >= 50%
share_half_cites = n_half_cites / n_pubs
# The all-records figures mix papers of very different ages. Within one publication year all papers
# have had the same time to be cited: the top-10% share of each year COHORT_FROM..MATURE_TO.
cohort_top10 = pd.Series({y: metrics.top_share(pubs.loc[pubs["year"] == y, "cited_by"], frac=0.10)
                          for y in range(COHORT_FROM, MATURE_TO + 1)})
cohort_top10_median, cohort_top10_min, cohort_top10_max = cohort_top10.agg(["median", "min", "max"])

# --- Citations by publication year (the citation-window effect)
by_year = (pubs.groupby("year")["cited_by"]
           .agg(papers="size", mean="mean", median="median", uncited=lambda s: (s == 0).mean())
           .reindex(range(int(pubs["year"].min()), LAST_FULL + 1)))
by_year["papers"] = by_year["papers"].fillna(0).astype(int)
enough = (by_year["papers"] >= 10)[::-1].cummin()[::-1]   # this and every later year have >= 10 papers
FIRST_YEAR = int(enough.idxmax())
n_before_first = int((pubs["year"] < FIRST_YEAR).sum())
max_before_first = int(by_year.loc[:FIRST_YEAR - 1, "papers"].max())
by_year = by_year.loc[FIRST_YEAR:].rename_axis("year").reset_index()
year_stats = by_year.set_index("year")
median_min, median_max = year_stats.loc[COHORT_FROM:MATURE_TO, "median"].agg(["min", "max"])


def period(label: str) -> pd.Series:
    """Citation statistics of the papers published in a period ("first–last" year)."""
    c = pubs.loc[pubs["year"].between(*periods.bounds(label)), "cited_by"]
    return pd.Series({"papers": len(c), "mean": c.mean(), "median": c.median(), "uncited": (c == 0).mean()})


p_prev, p_recent = period(P_PREV), period(P_RECENT)
y_before_last, y_last = year_stats.loc[LAST_FULL - 1], year_stats.loc[LAST_FULL]
# the median of the last two full years, below the range of COHORT_FROM..MATURE_TO (checked below)
median_a, median_b = fmt_count(y_before_last["median"]), fmt_count(y_last["median"])
if y_before_last["median"] < median_min:
    median_drop_text = f"The median falls to {median_a} for {LAST_FULL - 1} and {median_b} for {LAST_FULL}"
else:
    median_drop_text = f"The median is {median_a} for {LAST_FULL - 1} and falls to {median_b} for {LAST_FULL}"

peak = pubs.loc[cites.idxmax()]                                      # the most-cited paper
peak_year = int(peak["year"])
peak_year_share = peak["cited_by"] / cites[pubs["year"] == peak_year].sum()

# --- The most-cited publications
top = pubs.sort_values(["cited_by", "year", "eid"], ascending=[False, True, True]).head(15).copy()
top["doi_url"] = "https://doi.org/" + top["doi"]
top_cites = int(top["cited_by"].sum())
top_share = top_cites / total_cites
top_before_recent = int((top["year"] < RECENT_FROM).sum())
share_before_recent = (pubs["year"] < RECENT_FROM).mean()           # of all records
top_single = int((top["collab_type"] == "single-institution").sum())
top_intl = int((top["collab_type"] == "international").sum())
top_led = int((top["dece_first_author"] | top["dece_last_author"]).sum())   # a DECE first or last author

# --- Normalised citation score (NCS) by paper characteristics
in_window = pubs["year"].between(NCS_FROM, NCS_TO)
ncs = pubs[in_window & pubs["doc_type"].isin(CITABLE)].copy()
# Reviews are normalised together with articles: most years have too few reviews for an average of their
# own (a year with a single review would give it a score of 1 by construction).
ncs["ncs_type"] = ncs["doc_type"].replace(NCS_TYPE)
ncs["score"] = metrics.normalized_citation_score(ncs, by=["year", "ncs_type"])
ncs["score_year"] = metrics.normalized_citation_score(ncs, by="year")   # journal vs conference
n_ncs = len(ncs)
other_types = pubs.loc[in_window & ~pubs["doc_type"].isin(CITABLE), "doc_type"].value_counts()
n_ncs_other = int(other_types.sum())
n_reviews = int(ncs["doc_type"].eq("Review").sum())
reviews_year = ncs.loc[ncs["doc_type"].eq("Review"), "year"].value_counts().reindex(range(NCS_FROM, NCS_TO + 1),
                                                                                    fill_value=0)
intl = ncs["collab_type"] == "international"
led = ncs["dece_first_author"] | ncs["dece_last_author"]        # a DECE author is first or last author
# External-partner basis: counting only affiliations of co-authors who are not DECE authors, so that a DECE
# author's own second affiliation abroad (one that only DECE authors hold) does not make a paper international
intl_ext = ncs["is_international_external"]


def boot_mean_ci(values: pd.Series, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """Mean with a 95% percentile-bootstrap interval (papers resampled with replacement)."""
    x = values.dropna().to_numpy(dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(x), size=(n_boot, len(x)))
    lo, hi = np.quantile(x[idx].mean(axis=1), [0.025, 0.975])
    return float(x.mean()), float(lo), float(hi)


GROUPS = [  # (dimension, group, chart row label, mask of the group's papers, score column)
    ("Collaboration", "Single-institution", "Single-institution",
     ncs["collab_type"] == "single-institution", "score"),
    ("Collaboration", "National", "National", ncs["collab_type"] == "national", "score"),
    ("Collaboration", "International", "International", intl, "score"),
    # who leads the international papers: a DECE first or last author, or neither (partner-led)
    ("International papers", "International, DECE-led", "DECE-led", intl & led, "score"),
    ("International papers", "International, partner-led", "Partner-led", intl & ~led, "score"),
    # the international papers again on the external-partner basis (‡ in the caption)
    ("International, external‡", "International, external", "All", intl_ext, "score"),
    ("International, external‡", "International, external, DECE-led", "DECE-led", intl_ext & led, "score"),
    ("International, external‡", "International, external, partner-led", "Partner-led", intl_ext & ~led, "score"),
    ("DECE first or last author", "DECE first or last author", "Yes", led, "score"),
    ("DECE first or last author", "No DECE first or last author", "No", ~led, "score"),
    ("Open access", "Open access (all types)", "All types", ncs["oa_any"], "score"),
    ("Open access", "Gold", "Gold", ncs["oa_status"] == "gold", "score"),
    ("Open access", "Hybrid", "Hybrid", ncs["oa_status"] == "hybrid", "score"),
    ("Open access", "Bronze", "Bronze", ncs["oa_status"] == "bronze", "score"),
    # oa_status counts a paper once, in the order gold > hybrid > bronze > green: "Green" = repository copy only
    ("Open access", "Green", "Green only", ncs["oa_status"] == "green", "score"),
    ("Open access", "Closed", "Closed", ~ncs["oa_any"], "score"),
    ("Funding acknowledged", "Funding acknowledged", "Yes", ncs["has_funding"], "score"),
    ("Funding acknowledged", "No funding acknowledged", "No", ~ncs["has_funding"], "score"),
    ("Industry partner", "Industry partner", "Yes", ncs["has_industry_partner"], "score"),
    ("Industry partner", "No industry partner", "No", ~ncs["has_industry_partner"], "score"),
    ("Source type†", "Journal", "Journal", ncs["source_kind"] == "journal", "score_year"),
    ("Source type†", "Conference", "Conference", ncs["source_kind"] == "conference", "score_year"),
]
rows = []
for dim, group, row_label, mask, col in GROUPS:
    mean, lo, hi = boot_mean_ci(ncs.loc[mask, col])
    rows.append({"dimension": dim, "group": group, "row_label": row_label, "papers": int(mask.sum()),
                 "mean": mean, "lo": lo, "hi": hi, "median_score": ncs.loc[mask, col].median(),
                 "median_cites": ncs.loc[mask, "cited_by"].median()})
ncs_groups = pd.DataFrame(rows)
G = ncs_groups.set_index("group")
ncs_journal, ncs_conference = G.loc["Journal", "mean"], G.loc["Conference", "mean"]
journal_ratio = ncs_journal / ncs_conference


def ci_text(group: str, label: bool = True) -> str:
    """Mean score with its interval: '1.20 (95% CI 1.08–1.35)', or '1.20 (1.08–1.35)'."""
    r = G.loc[group]
    ci = f"{dp.fmt_num(r['lo'], 2)}–{dp.fmt_num(r['hi'], 2)}"
    return f"{dp.fmt_num(r['mean'], 2)} ({'95% CI ' if label else ''}{ci})"


def vs_average(group: str) -> str:
    """Mean score relative to the department average (1), e.g. '20% above'."""
    diff = G.loc[group, "mean"] - 1
    return f"{dp.fmt_pct(abs(diff), 0)} {'above' if diff >= 0 else 'below'}"


def vs_interval(group: str) -> str:
    """Position of the 95% interval relative to the department average."""
    r = G.loc[group]
    if r["lo"] > 1:
        return "above the average"
    if r["hi"] < 1:
        return "below the average"
    return "an interval that includes the average, so there is no clear difference"


# International papers: the advantage sits in the papers that partners lead
intl_led_median = G.loc["International, DECE-led", "median_score"]
intl_partner_median = G.loc["International, partner-led", "median_score"]
n_intl_partner = int(G.loc["International, partner-led", "papers"])
share_intl_partner = n_intl_partner / int(G.loc["International", "papers"])
mean_gap = G.loc["International, partner-led", "mean"] - G.loc["International, DECE-led", "mean"]
# share of the international papers' score above the average (score - 1, summed) that partner-led papers hold;
# the tail: the partner-led share of the highest-scoring 5% of the international papers, and the gap between
# the two groups without the highest-scoring 5% of each
intl_score, intl_partner = ncs.loc[intl, "score"], ~led[intl]
partner_excess_share = (intl_score - 1)[intl_partner].sum() / (intl_score - 1).sum()
partner_share_top5 = intl_partner[intl_score >= intl_score.quantile(0.95)].mean()
trimmed = {partner: s[s <= s.quantile(0.95)].mean() for partner, s in intl_score.groupby(intl_partner)}
gap_trimmed = trimmed[True] - trimmed[False]

# International papers on the external-partner basis: does the association hold without the papers that are
# international only through a DECE author's own second affiliation abroad?
n_intl, n_intl_ext = int(G.loc["International", "papers"]), int(G.loc["International, external", "papers"])
intl_dece_only = intl & ~intl_ext                 # international only through such second affiliations
n_intl_dece_only = int(intl_dece_only.sum())
ext_holds_text = ("so the association holds" if G.loc["International, external", "lo"] > 1
                  else "so the association is no longer clear")
# how many of the international papers remain international on that basis, in words (the numbers are not given,
# see below)
share_remain = n_intl_ext / n_intl
ext_remain_text = "almost all" if share_remain >= 0.95 else "most" if share_remain > 0.5 else "at most half"
# Small groups (no per-person metrics): a number of papers that are international only through DECE authors' own
# second affiliations abroad, or the score of such papers, is shown only when it is zero or rests on at least
# config.MIN_DECE_HOLDERS distinct DECE authors who hold these affiliations; with fewer it would in practice be one
# or two people's paper count or citation score. This covers the numbers a reader can derive as well: an
# external-partner group next to its address-based counterpart gives the number of its papers that are
# international only through such affiliations (its cell), and two cells give their difference. Other pages give
# the number of such papers for all countries together (by year, by period and for spans of years, papers of every
# type), and readers can combine the values of several pages, so the site gives these numbers only where
# dece_stats.small_groups publishes them. The papers compared here (three document types, NCS_FROM..NCS_TO) are
# not among them, so neither the figure nor the text gives the number of papers of an external-partner group or
# of a cell; the cells still decide which scores are shown. Author ids are only counted.
MIN_HOLDERS = config.MIN_DECE_HOLDERS
pub_aff = data.load_publication_affiliations()
authors = data.load_publication_authors()
own_abroad = pub_aff[pub_aff["eid"].isin(ncs.loc[intl_dece_only, "eid"]) & pub_aff["match_method"].ne("ignored")
                     & ~pub_aff["is_auth"] & pub_aff["held_only_by_dece"].fillna(False).astype(bool)
                     & pub_aff["country_code"].notna() & pub_aff["country_code"].ne(config.HOME_COUNTRY)]
own_abroad_holders = (own_abroad[["eid", "holder_author_pos"]]
                      .assign(author_pos=lambda d: d["holder_author_pos"].str.split(";")).explode("author_pos")
                      .astype({"author_pos": "int64"})
                      .merge(authors[["eid", "author_pos", "author_id", "is_dece"]], on=["eid", "author_pos"],
                             how="left", validate="many_to_one"))
assert own_abroad_holders["is_dece"].eq(True).all() and own_abroad_holders["author_id"].notna().all(), \
    "every holder of an own second affiliation is a DECE author with an author id"
assert own_abroad["eid"].nunique() == n_intl_dece_only, "each such paper has an own second affiliation abroad"
held_by = own_abroad_holders.groupby("eid")["author_id"].agg(set)   # each such paper: the DECE authors behind it


def rests_on(eids: frozenset) -> int:
    """Distinct DECE authors whose own second affiliations abroad make these papers international."""
    return len(set().union(*held_by.reindex(sorted(eids)).dropna()))


def remainders_safe(cells: list[frozenset]) -> bool:
    """Does every number a reader can form from these cells rest on no or at least MIN_HOLDERS DECE authors? The
    cells must be nested or disjoint; then each such number is a sum of remainders (a cell minus the cells inside
    it), and it is enough to check the remainders."""
    assert all(a <= b or b <= a or not a & b for a in cells for b in cells), "cells must be nested or disjoint"
    rests = [rests_on(c.difference(*[o for o in cells if o < c])) for c in cells]
    return all(n == 0 or n >= MIN_HOLDERS for n in rests)


# The external-partner groups of the figure (in order of importance), each with its address-based counterpart. The
# difference of their paper counts is the number of the group's papers that are international only through own
# affiliations abroad: the group's cell. A group (its score) is shown when its cell, with the cells of the groups
# shown before it, passes remainders_safe. A group that is not shown loses its score too: next to the score of its
# address-based counterpart, it would describe the papers of its cell (a weighted mean). Leadership splits the
# international papers, so both leadership groups are shown or neither (with one of them, the cell of the other
# follows from the total).
EXT_PAIRS = {"International, external": "International",
             "International, external, DECE-led": "International, DECE-led",
             "International, external, partner-led": "International, partner-led"}
EXT_SPLIT = ["International, external, DECE-led", "International, external, partner-led"]
MASKS = {group: mask for _, group, _, mask, _ in GROUPS}
assert all((MASKS[g] & ~MASKS[a]).sum() == 0 for g, a in EXT_PAIRS.items()), "external groups within their counterparts"
ext_cells = {g: frozenset(ncs.loc[MASKS[a] & ~MASKS[g], "eid"]) for g, a in EXT_PAIRS.items()}
assert set().union(*ext_cells.values()) <= set(held_by.index), "every paper of a cell has an own affiliation abroad"
ext_shown, kept = {}, []
for g, cell in ext_cells.items():
    ext_shown[g] = remainders_safe([*kept, cell])
    if ext_shown[g]:
        kept.append(cell)
assert ext_shown["International, external"], \
    "small groups: the text gives the scores of the international papers on the external-partner basis and of those " \
    "that are international only through own affiliations"
ext_split_shown = all(ext_shown[g] for g in EXT_SPLIT)
ext_hidden = [g for g in EXT_PAIRS if not ext_shown[g] or (g in EXT_SPLIT and not ext_split_shown)]
ncs_shown = ncs_groups[~ncs_groups["group"].isin(ext_hidden)].copy()     # the groups of the figure
# their number of papers (hover), except for the external-partner groups (see above)
ncs_shown["papers_text"] = [("not shown" if g in EXT_PAIRS else dp.fmt_int(n))
                            for g, n in zip(ncs_shown["group"], ncs_shown["papers"])]
only_mean, only_lo, only_hi = boot_mean_ci(ncs.loc[intl_dece_only, "score"])
dece_only_text = (f"score {dp.fmt_num(only_mean, 2)} ({dp.fmt_num(only_lo, 2)}–{dp.fmt_num(only_hi, 2)}), "
                  + ("an interval that includes the average" if only_lo <= 1 <= only_hi
                     else "above the average" if only_lo > 1 else "below the average"))
# The scores still bound the number of these papers (a weighted mean): n_intl * m_intl = (n_intl - n) * m_ext +
# n * m_only, where readers know n_intl (the hover) and each mean to the two decimals shown. counts_open: the
# numbers n that fit. They tell readers nothing new when the values the site publishes already place the papers of
# these years that are international only through own affiliations, of every type (n plus at most the n_ncs_other
# items of other types), between known[0] and known[1] (dece_stats.small_groups: the site's values for all
# countries together, with what readers know of those left out).
def shown_range(value: float) -> tuple[float, float]:
    """The values that round to the two decimals shown."""
    return round(value, 2) - 0.005, round(value, 2) + 0.005


(i_lo, i_hi), (e_lo, e_hi), (o_lo, o_hi) = (shown_range(m) for m in (G.loc["International", "mean"],
                                                                      G.loc["International, external", "mean"],
                                                                      only_mean))
counts_open = [n for n in range(n_intl + 1)
               if n_intl * i_lo - (n_intl - n) * e_hi - n * o_hi <= 0 <= n_intl * i_hi - (n_intl - n) * e_lo - n * o_lo]
rule = sg.site()
ALL_YEARS = range(int(pubs["year"].min()), PARTIAL + 1)
audit = sg.Audit(rule.moved, ALL_YEARS,
                 pd.crosstab(pubs["year"], pubs["collab_type"]).reindex(index=ALL_YEARS, columns=sg.COLLAB,
                                                                        fill_value=0),
                 rule.windows, sg.ranking_bounds(pubs, LAST_FULL)[1])
known = audit.range(audit.system(rule.split_shown, rule.cell_shown),
                    audit.vector(sg.INTL, range(NCS_FROM, NCS_TO + 1)))
if ext_split_shown:
    ext_lead_text = (f"On the external-partner basis, partner-led international papers again score higher: "
                     f"{ci_text('International, external, partner-led')} against "
                     f"{ci_text('International, external, DECE-led', False)} for DECE-led ones.")
else:
    ext_lead_text = ("On the external-partner basis, partner-led international papers again score higher than "
                     "DECE-led ones; these two scores are not shown, because with the other values they would "
                     "reveal a small group of papers (see the notes).")

# Overlap of the characteristics: international papers vs the others
intl_funded, other_funded = ncs.loc[intl, "has_funding"].mean(), ncs.loc[~intl, "has_funding"].mean()
intl_oa, other_oa = ncs.loc[intl, "oa_any"].mean(), ncs.loc[~intl, "oa_any"].mean()

# Mutually adjusted differences: least squares of the score on all characteristics at once
# (reference: single-institution, closed, no funding acknowledged, no industry partner); both models use
# the same bootstrap resamples of the papers
y = ncs["score"].to_numpy(dtype=float)
idx = np.random.default_rng(0).integers(0, n_ncs, size=(1000, n_ncs))


def adjusted(terms: dict[str, pd.Series]) -> pd.DataFrame:
    """Least-squares differences in mean score for indicator terms, with 95% bootstrap intervals."""
    X = np.column_stack([np.ones(n_ncs)] + [m.to_numpy(dtype=float) for m in terms.values()])
    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    boot = np.array([np.linalg.lstsq(X[i], y[i], rcond=None)[0] for i in idx])
    return pd.DataFrame({"diff": coef[1:], "lo": np.quantile(boot[:, 1:], 0.025, axis=0),
                         "hi": np.quantile(boot[:, 1:], 0.975, axis=0)}, index=list(terms))


TERMS = {"International": intl, "National": ncs["collab_type"] == "national",
         "Open access": ncs["oa_any"], "Funding acknowledged": ncs["has_funding"],
         "Industry partner": ncs["has_industry_partner"], "DECE first or last author": led}
adj = adjusted(TERMS)
# the same model with the partner indicators on the external-partner basis
adj_ext = adjusted({**TERMS, "International": intl_ext, "National": ncs["collab_type_external"] == "national",
                    "Industry partner": ncs["has_industry_partner_external"]})


def adj_text(term: str, table: pd.DataFrame | None = None) -> str:
    """Adjusted difference in mean score with its interval, e.g. '+0.22 (95% CI +0.06 to +0.40)'
    (from `adj`, or from `table`)."""
    r = (adj if table is None else table).loc[term]
    f = lambda v: ("+" if v >= 0 else "−") + dp.fmt_num(abs(v), 2)
    return f"{f(r['diff'])} (95% CI {f(r['lo'])} to {f(r['hi'])})"


def adj_word(term: str, table: pd.DataFrame | None = None) -> str:
    """'positive' / 'negative' / 'uncertain' from the adjusted 95% interval (of `adj`, or of `table`)."""
    r = (adj if table is None else table).loc[term]
    return "positive" if r["lo"] > 0 else "negative" if r["hi"] < 0 else "uncertain"


adj_summary = (f"With the other characteristics held fixed, the international difference is "
               f"{adj_word('International')}, the funding difference {adj_word('Funding acknowledged')}, "
               f"the open-access difference {adj_word('Open access')} and the difference for a DECE first or "
               f"last author {adj_word('DECE first or last author')}.")
ext_word = adj_word("International", adj_ext)
adj_ext_text = (f"With the international, national and industry-partner indicators on the external-partner basis, "
                f"the international difference is {adj_text('International', adj_ext)}, "
                f"{'also ' if ext_word == adj_word('International') else ''}{ext_word}.")
both_persist = all(adj_word(t) == "positive" for t in ("International", "Funding acknowledged"))
persist_text = ("both differences remain when these characteristics and DECE leadership are considered together"
                if both_persist else
                "not all of these differences remain clear when these characteristics and DECE leadership are "
                "considered together")


# Robustness: do the group medians order each comparison like the means?
ORDER_DIMS = {"collaboration type": ["Single-institution", "National", "International"],
              "open access vs closed": ["Open access (all types)", "Closed"],
              "funding": ["Funding acknowledged", "No funding acknowledged"],
              "industry partner": ["Industry partner", "No industry partner"],
              "source type": ["Journal", "Conference"],
              "DECE first or last author": ["DECE first or last author", "No DECE first or last author"],
              "leadership of the international papers": ["International, DECE-led", "International, partner-led"]}
same_order = [name for name, grp in ORDER_DIMS.items()
              if list(G.loc[grp, "mean"].sort_values().index)
              == list(G.loc[grp, "median_score"].sort_values().index)]
other_order = [name for name in ORDER_DIMS if name not in same_order]
if same_order and other_order:
    robust_text = (f"the group medians rank the groups as the means do for {join_and(same_order)}, "
                   f"but not for {join_and(other_order)}")
elif same_order:
    robust_text = "the group medians rank the groups as the means do for every comparison"
else:
    robust_text = "the group medians rank the groups differently from the means in every comparison"

# Matching coverage: industry partners rely on affiliations matched to ROR
cov_partner = meta["matching"]["excluding_auth"]["occurrence_matched_share"]
# Funding acknowledgements: papers before the latest ten full years against the latest five
funded_old = pubs.loc[pubs["year"] < TEN_FROM, "has_funding"].mean()
funded_recent = pubs.loc[pubs["year"].between(*periods.bounds(P_RECENT)), "has_funding"].mean()

# The text states these patterns of the current data: stop if new data change them.
checks = {
    "text: the windowed h-indices are lower than the h-index over all years": h_recent < h_ten < h_all,
    "text: n_h (>= h) papers are cited at least h times, but only n_h_next (< h + 1) at least h + 1 times":
        n_h >= h_all and n_h_next < h_next,
    "text: most papers have fewer than 10 citations": share_papers_lt10 > 0.5,
    "text: citations are concentrated (the most-cited 10% of the papers hold most of them)": top10_share > 0.5,
    "text: concentration is lower within publication years": cohort_top10_median < top10_share,
    "text: some years before NCS_FROM have fewer than NCS_MIN articles and reviews or conference papers": thin.any(),
    "text: recent papers average fewer citations than those of the period before": p_recent["mean"] < p_prev["mean"],
    "text: the median falls in the latest full year, below that of every year COHORT_FROM..MATURE_TO":
        y_last["median"] < median_min and y_last["median"] <= y_before_last["median"],
    "text: the mean jumps in the year of the most-cited paper (shown in the chart, above both neighbours)":
        FIRST_YEAR < peak_year < LAST_FULL
        and year_stats.loc[peak_year, "mean"] > year_stats.loc[[peak_year - 1, peak_year + 1], "mean"].max(),
    "text: most of the top papers are older than the latest period, more than among all papers":
        top_before_recent / len(top) > max(0.5, share_before_recent),
    "caption: every title in the top-papers table links to its DOI": top["doi"].notna().all(),
    "text: the other items in the window are mostly book chapters, also editorials, errata and other types":
        other_types.index[0] == "Book chapter" and {"Editorial", "Erratum"} <= set(other_types.index)
        and len(other_types) > 3,
    "text: most years have too few reviews for an average of their own": (reviews_year < NCS_MIN).mean() > 0.5,
    "text: international papers are cited above the average": G.loc["International", "lo"] > 1,
    "text: mainly those led by partners (most of the international papers' excess citations)":
        partner_excess_share > 0.5,
    "text: papers with a funding acknowledgement are cited above the average": G.loc["Funding acknowledged", "lo"] > 1,
    "text: partner-led international papers score higher than DECE-led ones": mean_gap > 0,
    "text: on the external-partner basis too, partner-led international papers score higher":
        G.loc["International, external, partner-led", "mean"] > G.loc["International, external, DECE-led", "mean"],
    "small groups: every external-partner value of the figure and the text, and every number a reader can form "
    "from them and the address-based values, rests on no or at least MIN_HOLDERS DECE authors":
        remainders_safe([ext_cells[g] for g in EXT_PAIRS
                         if g in set(ncs_shown["group"]) or g == "International, external"
                         or (g in EXT_SPLIT and ext_split_shown)]),
    "small groups: no external-partner group comes with its number of papers (next to the address-based number it "
    "would give how many papers of NCS_FROM..NCS_TO are international only through own affiliations, which is not "
    "a value the site publishes)":
        ncs_shown.loc[ncs_shown["group"].isin(EXT_PAIRS), "papers_text"].eq("not shown").all(),
    "small groups: the scores shown do not narrow that number beyond what the values the site publishes give":
        bool(counts_open) and min(counts_open) <= known[0] and max(counts_open) + n_ncs_other >= known[1],
    "small groups: the site's rule for the values for all countries together holds": all(rule.checks().values()),
    "text: the medians of DECE-led and partner-led international papers differ much less than their means":
        abs(intl_partner_median - intl_led_median) < mean_gap / 3,
    "text: most of the difference comes from a tail of highly cited partner-led papers (over-represented in the "
    "top 5% of the international papers; without the top 5% of each group, less than half of the gap remains)":
        partner_share_top5 > share_intl_partner and gap_trimmed < mean_gap / 2,
    "text: the open-access routes differ, green-only above gold": G.loc["Green", "lo"] > G.loc["Gold", "hi"],
    "text: journal papers are cited far more than conference papers":
        journal_ratio > 2 and G.loc["Journal", "lo"] > G.loc["Conference", "hi"],
    "text: international papers more often acknowledge funding and are more often open access":
        intl_funded > other_funded and intl_oa > other_oa,
    "notes: funding acknowledgements are missing especially for older papers": funded_old < funded_recent,
}
failed = [claim for claim, ok in checks.items() if not ok]
assert not failed, f"the text no longer matches the data: {failed}"

How often are the department's publications cited, how evenly are the citations spread across papers, which papers are cited most, and which kinds of papers attract more citations than comparable ones? This page answers these questions with the Scopus “Cited by” count of each of the {eval}`dp.fmt_int(n_pubs)` records, a snapshot taken on {eval}`AS_OF_TEXT`. The counts include self-citations and only citations from documents indexed in Scopus.

In [ ]:
dp.kpi_html([
    dp.KPI("Citations", total_cites, f"Scopus “Cited by” on {AS_OF_TEXT}"),
    dp.KPI("Mean per publication", dp.fmt_num(mean_cites, 1), f"all {dp.fmt_int(n_pubs)} records"),
    dp.KPI("Median per publication", fmt_count(median_cites), f"all {dp.fmt_int(n_pubs)} records"),
    dp.KPI("Not yet cited", dp.fmt_pct(share_uncited),
           f"{dp.fmt_int(n_uncited)} papers; {dp.fmt_pct(share_uncited_mature)} of the papers up to {MATURE_TO}"),
    dp.KPI("h-index", h_all, "all years"),
    dp.KPI(f"h-index {TEN_YEARS}", h_ten, f"papers published {TEN_YEARS}"),
    dp.KPI(f"h-index {P_RECENT}", h_recent, f"papers published {P_RECENT}"),
    dp.KPI("Publications with ≥ 100 citations", n_100,
           f"{dp.fmt_pct(share_100)} of the papers, {dp.fmt_pct(share_cites_100)} of the citations"),
], min_width=140)

:::{important} Key findings
- By {eval}`AS_OF_TEXT` the {eval}`dp.fmt_int(n_pubs)` publications had received {eval}`dp.fmt_int(total_cites)` citations, a mean of {eval}`dp.fmt_num(mean_cites, 1)` and a median of {eval}`fmt_count(median_cites)` per paper. The department's h-index is {eval}`str(h_all)` over all years and {eval}`str(h_ten)` for the papers published {eval}`TEN_YEARS`.
- Citations are concentrated: the most-cited 1% of the papers hold {eval}`dp.fmt_pct(top1_share)` of all citations and the most-cited 10% hold {eval}`dp.fmt_pct(top10_share)` (all records, of mixed ages; within each publication year from {eval}`str(COHORT_FROM)` to {eval}`str(MATURE_TO)` the most-cited 10% hold a median {eval}`dp.fmt_pct(cohort_top10_median)` of that year's citations), while {eval}`dp.fmt_pct(share_uncited_mature)` of the papers published up to {eval}`str(MATURE_TO)` have not been cited.
- Recent papers have had less time to be cited: papers from {eval}`P_PREV` average {eval}`dp.fmt_num(p_prev['mean'], 1)` citations, papers from {eval}`P_RECENT` {eval}`dp.fmt_num(p_recent['mean'], 1)`, and {eval}`dp.fmt_pct(y_last['uncited'])` of the {eval}`str(LAST_FULL)` papers are still uncited.
- Compared with papers of the same year and type ({eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`), internationally co-authored papers are cited {eval}`vs_average('International')` the department average, mainly those led by partners: the mean normalised score is {eval}`ci_text('International, DECE-led')` for international papers with a DECE first or last author and {eval}`ci_text('International, partner-led', False)` for those where DECE holds neither position. Counting only affiliations of co-authors who are not DECE authors, international papers score {eval}`ci_text('International, external')`, {eval}`ext_holds_text`. Papers with a funding acknowledgement are cited {eval}`vs_average('Funding acknowledged')` the average; {eval}`persist_text`.
- Journal papers are cited far more than conference papers: relative to all articles, reviews and conference papers of the same year ({eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`), they score {eval}`dp.fmt_num(ncs_journal, 2)` against {eval}`dp.fmt_num(ncs_conference, 2)`, a factor of {eval}`dp.fmt_num(journal_ratio, 1)`.
:::

## Headline indicators

The h-index is the largest number *h* such that *h* papers have at least *h* citations each. The department's h-index is {eval}`str(h_all)`: {eval}`dp.fmt_int(n_h)` of its papers have been cited at least {eval}`str(h_all)` times, but only {eval}`dp.fmt_int(n_h_next)` at least {eval}`str(h_next)` times. The windowed h-indices count only the papers published in those years (with their citations up to {eval}`AS_OF_TEXT`), so they are lower partly because they cover fewer papers and fewer years of citations. The mean is {eval}`dp.fmt_num(mean_to_median, 1)` times the median because a small number of papers are cited very often, as the next section shows.

In [ ]:
#| label: cell-impact-distribution
fig = go.Figure()
for share, count, name, color in [("share_papers", "papers", "Publications", dp.CATEGORICAL[0]),
                                  ("share_citations", "citations", "Citations", dp.CATEGORICAL[1])]:
    fig.add_bar(x=dist["cite_class"], y=dist[share], name=name, marker_color=color,
                customdata=dist[[count]].to_numpy(),
                hovertemplate=f"{name}: %{{y:.1%}} (%{{customdata[0]:,}})<extra></extra>")
fig.update_layout(barmode="group", hovermode="x unified", margin=dict(t=24))
fig.update_xaxes(title_text="Citations per publication")
fig.update_yaxes(title_text="Share of all publications or citations", tickformat=".0%")
fig.show()

## How citations are distributed

:::{figure} #cell-impact-distribution
:label: fig-impact-distribution
Share of all papers and of all citations in each class of citation counts (all {eval}`dp.fmt_int(n_pubs)` records; Scopus counts on {eval}`AS_OF_TEXT`).
:::

Most papers are cited modestly ([](#fig-impact-distribution)): {eval}`dp.fmt_pct(share_papers_lt10)` have fewer than 10 citations, and together they account for only {eval}`dp.fmt_pct(share_cites_lt10)` of all citations. At the other end, the {eval}`dp.fmt_int(n_100)` papers with at least 100 citations ({eval}`dp.fmt_pct(share_100)` of the papers) collect {eval}`dp.fmt_pct(share_cites_100)` of the citations.

In [ ]:
#| label: cell-impact-concentration
k = np.unique(np.linspace(0, n_pubs, 401).round().astype(int))
fig = go.Figure()
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=dp.OTHER_COLOR, width=1),
                hoverinfo="skip", name="Equal citations for every paper")
fig.add_scatter(x=k / n_pubs, y=cum_share[k], mode="lines", line=dict(color=dp.CATEGORICAL[0]),
                name="DECE papers",
                hovertemplate="Most-cited %{x:.1%} of the papers: %{y:.1%} of the citations<extra></extra>")
marks = [(top1_n, "Top 1%"), (top10_n, "Top 10%"), (half_n, "Top 50%")]
fig.add_scatter(x=[n / n_pubs for n, _ in marks], y=[cum_share[n] for n, _ in marks], mode="markers",
                marker=dict(color=dp.CATEGORICAL[0], size=9), customdata=[label for _, label in marks],
                hovertemplate="%{customdata} of the papers: %{y:.1%} of the citations<extra></extra>")
for n, label in marks:
    fig.add_annotation(x=n / n_pubs, y=cum_share[n], text=f"{label}: {dp.fmt_pct(cum_share[n])}",
                       xanchor="left", yanchor="top", xshift=8, yshift=-2, font=dict(size=12))
fig.add_annotation(x=0.66, y=0.6, text="Equal citations for every paper", xanchor="left", yanchor="top",
                   font=dict(size=11))
fig.update_xaxes(title_text="Share of publications, most cited first", tickformat=".0%", range=[0, 1.0],
                 showgrid=True, zeroline=False)
fig.update_yaxes(title_text="Share of all citations", tickformat=".0%", range=[0, 1.04], zeroline=False)
fig.update_layout(showlegend=False, height=380, margin=dict(t=16))
fig.show()

:::{figure} #cell-impact-concentration
:label: fig-impact-concentration
Concentration of citations: the cumulative share of all citations held by the most-cited papers. The grey diagonal is what equal citation of every paper would give.
:::

[](#fig-impact-concentration) ranks the papers from most to least cited. The top 1% ({eval}`dp.fmt_int(top1_n)` papers) hold {eval}`dp.fmt_pct(top1_share)` of all citations, the top 10% ({eval}`dp.fmt_int(top10_n)` papers) hold {eval}`dp.fmt_pct(top10_share)` and the more-cited half {eval}`dp.fmt_pct(half_share)`; only {eval}`dp.fmt_int(n_half_cites)` papers ({eval}`dp.fmt_pct(share_half_cites)`) account for half of all citations. The Gini coefficient is {eval}`dp.fmt_num(gini, 2)` (0 = every paper cited equally, 1 = all citations on one paper). Part of this concentration comes from mixing papers of different ages, since older papers have had longer to be cited: within each publication year from {eval}`str(COHORT_FROM)` to {eval}`str(MATURE_TO)`, the most-cited 10% of that year's papers hold between {eval}`dp.fmt_pct(cohort_top10_min)` and {eval}`dp.fmt_pct(cohort_top10_max)` of its citations (median {eval}`dp.fmt_pct(cohort_top10_median)`). Such skew is normal for citation data, which is why the comparisons below also report medians and confidence intervals.

In [ ]:
#| label: cell-impact-by-year
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.6, 0.4])
papers = by_year[["papers"]].to_numpy()
fig.add_scatter(x=by_year["year"], y=by_year["mean"], name="Mean", mode="lines",
                line_color=dp.CATEGORICAL[0], customdata=papers,
                hovertemplate="Mean: %{y:.1f} citations (%{customdata[0]:,} papers)<extra></extra>",
                row=1, col=1)
fig.add_scatter(x=by_year["year"], y=by_year["median"], name="Median", mode="lines",
                line_color=dp.CATEGORICAL[1], hovertemplate="Median: %{y:.1f} citations<extra></extra>",
                row=1, col=1)
fig.add_scatter(x=by_year["year"], y=by_year["uncited"], name="Not yet cited", mode="lines",
                line_color=dp.CATEGORICAL[2], showlegend=False,
                hovertemplate="Not yet cited: %{y:.1%}<extra></extra>", row=2, col=1)
fig.update_yaxes(title_text="Citations per publication", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="Publications not yet cited", tickformat=".0%", rangemode="tozero", row=2, col=1)
fig.update_layout(height=560, hovermode="x unified", margin=dict(t=24))
dp.style_year_axis(fig)
fig.show()

## Citations and publication age

:::{figure} #cell-impact-by-year
:label: fig-impact-by-year
Mean and median citations per paper (top) and share of papers not yet cited (bottom) by publication year, {eval}`str(FIRST_YEAR)`–{eval}`str(LAST_FULL)` (earlier years have at most {eval}`str(max_before_first)` papers each, {eval}`dp.fmt_int(n_before_first)` in total). {eval}`str(PARTIAL)` (partial year) is not shown.
:::

Citations take a few years to build up, so raw counts understate the most recent papers ([](#fig-impact-by-year)). For each publication year from {eval}`str(COHORT_FROM)` to {eval}`str(MATURE_TO)` the median paper has between {eval}`fmt_count(median_min)` and {eval}`fmt_count(median_max)` citations, and of all papers published up to {eval}`str(MATURE_TO)`, {eval}`dp.fmt_pct(share_uncited_mature)` are uncited. Papers from {eval}`P_PREV` average {eval}`dp.fmt_num(p_prev['mean'], 1)` citations and those from {eval}`P_RECENT` {eval}`dp.fmt_num(p_recent['mean'], 1)`. {eval}`median_drop_text`, a year in which {eval}`dp.fmt_pct(y_last['uncited'])` of the papers are still uncited. The lower values of recent years reflect a shorter citation window, not necessarily lower impact. The mean also jumps with single highly cited papers: the most-cited paper ({eval}`str(peak_year)`) alone holds {eval}`dp.fmt_pct(peak_year_share)` of the citations to that year's papers.

In [ ]:
#| label: cell-impact-top-papers
dp.table_html(
    top,
    columns={"year": "Year", "title": "Title", "source_title": "Venue", "doc_type": "Type",
             "cited_by": "Citations"},
    links={"title": "doi_url"},
)

## Most-cited publications

:::{figure} #cell-impact-top-papers
:kind: table
:label: tbl-impact-top-papers
The 15 most-cited publications (Scopus counts on {eval}`AS_OF_TEXT`; titles link to the DOI).
:::

The 15 papers in [](#tbl-impact-top-papers) have received {eval}`dp.fmt_int(top_cites)` citations, {eval}`dp.fmt_pct(top_share)` of the department's total. {eval}`str(top_before_recent)` of them were published before {eval}`str(RECENT_FROM)`, in line with the citation-window effect above. Of the 15, {eval}`str(top_single)` are single-institution papers and {eval}`str(top_intl)` have an affiliation abroad; in {eval}`str(top_led)` a DECE author is the first or the last author.

In [ ]:
#| label: cell-impact-ncs
# one row per group under a bold header row per dimension (short labels keep the chart usable on phones); only
# the groups that may be shown (small groups), with the scores rounded as in the hover and no number of papers for
# the external-partner groups
d = ncs_shown.copy()
d[["mean", "lo", "hi", "median_score"]] = d[["mean", "lo", "hi", "median_score"]].round(2)
tickvals, ticktext, ypos = [], [], []
for dim, rows_dim in d.groupby("dimension", sort=False):
    tickvals.append(len(tickvals))
    ticktext.append(f"<b>{dim}</b>")
    for label in rows_dim["row_label"]:
        ypos.append(len(tickvals))
        tickvals.append(len(tickvals))
        ticktext.append(label)
d["y"] = ypos
d["name"] = d["dimension"].str.rstrip("†‡") + ": " + d["row_label"]
fig = go.Figure()
fig.add_vline(x=1, line=dict(color=dp.INK["muted"], width=1))
fig.add_scatter(
    x=d["mean"], y=d["y"], mode="markers", marker=dict(color=dp.CATEGORICAL[0], size=9),
    error_x=dict(type="data", symmetric=False, array=d["hi"] - d["mean"], arrayminus=d["mean"] - d["lo"],
                 color=dp.CATEGORICAL[0], thickness=1.5, width=0),
    customdata=d[["name", "lo", "hi", "papers_text", "median_cites", "median_score"]].to_numpy(),
    hovertemplate=("<b>%{customdata[0]}</b><br>Mean score: %{x:.2f} "
                   "(95% CI %{customdata[1]:.2f}–%{customdata[2]:.2f})<br>"
                   "Median score: %{customdata[5]:.2f}<br>Papers: %{customdata[3]}<br>"
                   "Median citations: %{customdata[4]:.1f}<extra></extra>"),
)
fig.add_annotation(x=1, y=1, xref="x", yref="paper", yanchor="bottom", xanchor="left", xshift=4,
                   text="Department average", font=dict(size=11))
fig.update_xaxes(title_text="Mean normalised citation score",
                 range=[0, float(np.ceil(d["hi"].max() * 4) / 4)], showgrid=True, zeroline=False)
fig.update_yaxes(tickvals=tickvals, ticktext=ticktext, range=[len(tickvals) - 0.4, -0.6],
                 showgrid=False, showline=False, zeroline=False)
fig.update_layout(height=dp.fig_height(len(tickvals), per_row=22), showlegend=False, margin=dict(t=28))
fig.show()

## Normalised impact by paper characteristics

Raw counts mix the effects of age and document type. The normalised citation score divides the citations of each paper by the mean citations of the department's papers of the same publication year and type: a score of 1 is the department average for comparable papers and 1.5 means 50% more citations than that average. Conference papers are compared with conference papers; articles and reviews are compared together, because most years have too few reviews ({eval}`dp.fmt_int(n_reviews)` in these years) for a yearly average of their own. The comparison uses the {eval}`dp.fmt_int(n_ncs)` articles, reviews and conference papers published {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`. Earlier years are left out because some of them have fewer than {eval}`str(NCS_MIN)` articles and reviews or fewer than {eval}`str(NCS_MIN)` conference papers, later ones because their papers have had too little time to be cited; the {eval}`dp.fmt_int(n_ncs_other)` items of other types in these years (book chapters, editorials, errata and others) are also left out.

:::{figure} #cell-impact-ncs
:label: fig-impact-ncs
Mean normalised citation score by group with 95% bootstrap confidence intervals ({eval}`dp.fmt_int(n_ncs)` articles, reviews and conference papers, {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`). The vertical line is the department average. International papers are *DECE-led* when a DECE author is first or last author and *partner-led* when DECE holds neither position. ‡ *External*: counting only affiliations of co-authors who are not DECE authors, so that a DECE author's own second affiliation abroad does not make a paper international (number of papers not shown, see the notes). A paper with several open-access routes counts once, in the first of gold, hybrid, bronze and green. † Source type is normalised by publication year only: with document type in the normalisation both groups would score 1 by construction.
:::

[](#fig-impact-ncs) shows associations, not causal effects:

- **Collaboration.** Internationally co-authored papers score {eval}`ci_text('International')`, single-institution papers {eval}`ci_text('Single-institution', False)` and papers whose only partners are other Greek organisations {eval}`ci_text('National', False)`. A DECE author's own second affiliation abroad (a joint, adjunct or visiting position) also makes a paper international. Counting only affiliations of co-authors who are not DECE authors (the external-partner basis, ‡ in the figure), {eval}`ext_remain_text` of the international papers remain international; they score {eval}`ci_text('International, external')`, {eval}`ext_holds_text`. The papers that are international only through such a second affiliation {eval}`dece_only_text`.
- **Leadership.** International papers with a DECE first or last author score {eval}`ci_text('International, DECE-led')`, those led by partners ({eval}`dp.fmt_int(n_intl_partner)` papers, {eval}`dp.fmt_pct(share_intl_partner)` of the international ones) {eval}`ci_text('International, partner-led', False)`. Their medians differ much less ({eval}`dp.fmt_num(intl_led_median, 2)` and {eval}`dp.fmt_num(intl_partner_median, 2)`), so most of the difference comes from a tail of highly cited papers led by partners. {eval}`ext_lead_text` Over all papers, those with a DECE first or last author score {eval}`ci_text('DECE first or last author')`, the others {eval}`ci_text('No DECE first or last author', False)`.
- **Open access.** Open-access papers score {eval}`ci_text('Open access (all types)')` against {eval}`ci_text('Closed')` for closed ones, but the routes differ: green-only open access (free only as a repository copy) scores {eval}`ci_text('Green')`, while gold open access (papers in fully open-access journals) scores {eval}`ci_text('Gold')`, {eval}`vs_interval('Gold')`.
- **Funding.** Papers that acknowledge funding score {eval}`ci_text('Funding acknowledged')`, papers without an acknowledgement {eval}`ci_text('No funding acknowledged')`.
- **Industry.** The {eval}`dp.fmt_int(G.loc['Industry partner', 'papers'])` papers with an industry partner score {eval}`ci_text('Industry partner')`, {eval}`vs_interval('Industry partner')`.
- **Source type.** Relative to all articles, reviews and conference papers of the same year, journal papers score {eval}`ci_text('Journal')` and conference papers {eval}`ci_text('Conference')`, a factor of {eval}`dp.fmt_num(journal_ratio, 1)`.

These characteristics overlap: {eval}`dp.fmt_pct(intl_funded)` of the international papers acknowledge funding and {eval}`dp.fmt_pct(intl_oa)` are open access, against {eval}`dp.fmt_pct(other_funded)` and {eval}`dp.fmt_pct(other_oa)` of the other papers. A least-squares model of the score on collaboration type, open access, funding, industry partnership and DECE leadership (a DECE first or last author) together (bootstrap 95% intervals) estimates each difference with the others held fixed. Compared with single-institution papers, international co-authorship is associated with a difference in mean score of {eval}`adj_text('International')` and national co-authorship with {eval}`adj_text('National')`. A funding acknowledgement is associated with {eval}`adj_text('Funding acknowledged')`, open access with {eval}`adj_text('Open access')`, an industry partner with {eval}`adj_text('Industry partner')` and a DECE first or last author with {eval}`adj_text('DECE first or last author')`. {eval}`adj_summary` {eval}`adj_ext_text` Research area and venue, which the score does not normalise, can still explain part of all these differences.

## Notes & caveats

- **Snapshot.** Citation counts are the Scopus “Cited by” values on {eval}`AS_OF_TEXT`. They include self-citations, count only citing documents indexed in Scopus, and keep growing; other databases (Web of Science, Google Scholar) give different counts.
- **Partial year.** {eval}`str(PARTIAL)` covers records up to {eval}`AS_OF_TEXT` ({eval}`dp.fmt_int(n_partial)` papers). They are included in the all-records statistics (totals, distribution, concentration), but not in the citations by publication year ([](#fig-impact-by-year)), the windowed h-indices, the period averages or the normalised scores.
- **Department average, not world average.** The normalised score compares DECE papers with other DECE papers of the same year and type (articles and reviews together, conference papers apart). It is not a field-normalised indicator such as the field-weighted citation impact (FWCI) and says nothing about impact relative to the world. Research areas with different citation habits are not separated. Because the score averages 1 in every year by construction, it cannot show whether impact rose or fell over time.
- **Skewed data.** Means depend on a few highly cited papers; the bootstrap intervals show this uncertainty. As a check, {eval}`robust_text` (medians in the hover of [](#fig-impact-ncs)).
- **Flags.** Open-access flags describe the status at the export date, not at publication. Funding acknowledgements in Scopus are incomplete, especially for older papers, so some papers counted without funding were funded. Industry partners are found through affiliations matched to ROR organisations of type company. {eval}`dp.fmt_pct(cov_partner)` of the affiliation occurrences outside AUTH are matched to a ROR organisation, and many companies have no ROR record, so this group is undercounted. DECE first and last authors are identified from each author's affiliation text (see [Research community](people.ipynb)), and author order is an imperfect signal of who led a paper.
- **Definitions.** “Type” is the Scopus document type. The source type (journal, conference, book) is derived from the document type, not from the venue, so a conference paper published in a journal issue counts as conference, as on [Research output](output.ipynb). *International* papers have at least one affiliation in a country other than Greece, read from the affiliation addresses as SciVal and the Leiden Ranking do; the external-partner variant (‡ in [](#fig-impact-ncs)) counts only affiliations of co-authors who are not DECE authors. Collaboration types, open-access categories and the other definitions are described in [Methodology & data](methodology.ipynb).
- **Small groups.** A number of papers that are international only through DECE authors' own second affiliations abroad, or a score of such papers, is shown only when at least {eval}`str(MIN_HOLDERS)` DECE authors hold these affiliations, since with fewer it would in practice describe one or two people. This also covers numbers that could be derived from the values shown, on this page or together with those of other pages, such as the difference between the address-based and the external-partner count of the same group. The site gives the number of these papers only for the years, periods and spans of years chosen for all its pages together (see [Methodology & data](methodology.ipynb)); the articles, reviews and conference papers of {eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)` compared here are not among them, so the external-partner groups are shown without their number of papers.